In [ ]:
# InfoMax vs. Baseline Transformer on a Synthetic Disentanglement Task

# ---- Imports ----

import os
import math
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---- Configuration ----

DEVICE      = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
SEED        = 42
SEQ_LEN     = 30    # since 3 × 10 for each composite token
VOCAB_SIZE  = 60
NUM_CLASSES = 10
D_MODEL     = 256
NUM_HEADS   = 4
NUM_LAYERS  = 3
FF_DIM      = 1024
DROPOUT     = 0.1
BATCH_SIZE  = 64
NUM_EPOCHS  = 30
PATIENCE    = 5
SAVE_PATH   = Path("./attention_maps")
SAVE_PATH.mkdir(exist_ok=True)

REALTIME_PLOTS = True  # Set to False to disable real-time plots

# ---- Utils ----
torch.manual_seed(SEED)
np.random.seed(SEED)

def generate_vocab():
    return {
        'colors': [f"color{i}" for i in range(10)],
        'shapes': [f"shape{i}" for i in range(10)],
        'positions': [f"pos{i}" for i in range(10)]
    }

VOCAB       = generate_vocab()
TOKEN_TO_ID = {}
index       = 0
for group in VOCAB.values():
    for tok in group:
        TOKEN_TO_ID[tok] = index
        index += 1
ID_TO_TOKEN = {i: tok for tok, i in TOKEN_TO_ID.items()}

# ---- Dataset ----
class SyntheticDisentangleDataset(Dataset):
    def __init__(self, size):
        self.size  = size
        self.vocab = VOCAB
        self.data  = []
        for _ in range(size):
            tokens = []
            target = []
            for _ in range(SEQ_LEN):
                color = np.random.choice(self.vocab['colors'])
                shape = np.random.choice(self.vocab['shapes'])
                pos   = np.random.choice(self.vocab['positions'])
                tokens.append((color, shape, pos))
                target.append(TOKEN_TO_ID[color])  # reverse color prediction task
            self.data.append((tokens, target[::-1]))

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        tokens, target = self.data[idx]
        input_ids      = []
        for color, shape, pos in tokens:
            input_ids.append(TOKEN_TO_ID[color])
            input_ids.append(TOKEN_TO_ID[shape])
            input_ids.append(TOKEN_TO_ID[pos])
        
        assert len(input_ids) % 3 == 0, f"Expected triplets, got {len(input_ids)} tokens"
        return torch.tensor(input_ids), torch.tensor(target)

train_ds     = SyntheticDisentangleDataset(5000)
val_ds       = SyntheticDisentangleDataset(1000)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)

# ---- Positional Encoding ----
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe          = torch.zeros(max_len, d_model)
        position    = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term    = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe     = pe.unsqueeze(0)  # [1, max_len, d_model]

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)].to(x.device)
        return x

# ---- Transformer Block with Pre-Norm ----
class TransformerBlock(nn.Module):
    def __init__(self, d_model, heads, ff_dim, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=heads, dropout=dropout, batch_first=True)
        self.ln1  = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model),
        )
        self.ln2     = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        attn_input = self.ln1(x)
        attn_output, attn_weights = self.attn(attn_input, attn_input, attn_input, need_weights=True, average_attn_weights=False, attn_mask=attn_mask)

        # Residual Pre-norm
        x        = x + self.dropout(attn_output)
        ff_input = self.ln2(x)
        x        = x + self.dropout(self.ff(ff_input))
        return x, attn_weights

# ---- Baseline Transformer ----
class BaselineTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, ff_dim, dropout):
        super().__init__()
        self.embedding    = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
        self.layers       = nn.ModuleList([
            TransformerBlock(d_model, nhead, ff_dim, dropout) for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(d_model, NUM_CLASSES)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        attn_maps = []
        for layer in self.layers:
            x, attn = layer(x)
            attn_maps.append(attn)
        logits = self.classifier(x)
        return logits, attn_maps

# ---- InfoMax Transformer ----
class InfoMaxTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, ff_dim, dropout):
        super().__init__()
        self.embedding    = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
        self.layers       = nn.ModuleList([
            TransformerBlock(d_model, nhead, ff_dim, dropout) for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(d_model, NUM_CLASSES)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        attn_maps = []
        layer_outputs = []
        for i, layer in enumerate(self.layers):
            x, attn = layer(x)
            attn_maps.append(attn)
            if i == len(self.layers) - 1:
                layer_outputs.append(x)
        logits = self.classifier(x)
        return logits, attn_maps, layer_outputs
    
    def compute_orthogonality_loss(self, head_output):
        """
        Penalize similarity between per-head output representations.
        head_output: Tensor of shape [B, T, D] where D = H * d_h
        """
        B, T, D = head_output.shape
        H = NUM_HEADS
        d_h = D // H
        head_repr = head_output.view(B, T, H, d_h).transpose(1, 2)  # [B, H, T, d_h]

        # Mean over tokens
        H_out = head_repr.mean(dim=2)           # [B, H, d_h]
        H_out = F.normalize(H_out, dim=-1)      # unit length per head
        gram = torch.einsum('bhd,bkd->bhk', H_out, H_out)  # [B, H, H]
        identity = torch.eye(H, device=gram.device)[None]  # [1, H, H]
        return ((gram - identity) ** 2).mean()      # scalar
    
    def compute_auxiliary_losses(self, attn_weights, head_outputs, entropy_weight=1.0, orthogonality_weight=1.0):
        # Entropy of attention weights
        def softmax_entropy(attn):
            p = attn.clamp(min=1e-9)
            return -(p * p.log()).sum(dim=-1).mean()
        
        entropy_loss       = 0
        orthogonality_loss = 0
        
        if attn_weights:
            for A in attn_weights[-1]:  # last layer only
                entropy_loss += softmax_entropy(A)
            entropy_loss = -entropy_loss / len(attn_weights[-1])  # maximize entropy
        
        if head_outputs:
            H                  = head_outputs[0]  # shape: [B, T, D]
            orthogonality_loss = self.compute_orthogonality_loss(H)
        
        return entropy_weight * entropy_loss + orthogonality_weight * orthogonality_loss

# ---- Visualization Utility ----
def plot_metrics(history, title="Training Metrics", save_path=None):
    if not REALTIME_PLOTS:
        return
    plt.figure(figsize=(12, 6))
    for key, values in history.items():
        plt.plot(values, label=key)
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Value")
    plt.legend()
    if save_path:
        plt.savefig(save_path)
    plt.show()

def visualize_attention_heads(attn_map, sample_idx=0, layer_idx=0, save_path=None):
    """
    Visualize attention maps for all heads of a given layer and sample.
    attn_map: Tensor of shape [B, H, T, T] or [H, T, T]
    """
    if attn_map.ndim == 4:
        # Standard: [B, H, T, T]
        attn = attn_map[sample_idx].detach().cpu().numpy()  # shape [H, T, T]
    elif attn_map.ndim == 3:
        # Possibly already [H, T, T]
        attn = attn_map.detach().cpu().numpy()
    else:
        raise ValueError(f"Expected attention map of dim 3 or 4, got {attn_map.ndim}")

    num_heads = attn.shape[0]
    fig, axes = plt.subplots(1, num_heads, figsize=(3 * num_heads, 3))
    fig.suptitle(f"Layer {layer_idx} Attention Heads for Sample {sample_idx}")

    for h in range(num_heads):
        ax = axes[h] if num_heads > 1 else axes
        ax.imshow(attn[h], cmap='viridis', aspect='auto')
        ax.set_title(f"Head {h}")
        ax.set_xlabel("Key Pos")
        ax.set_ylabel("Query Pos")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
    plt.show()

# ---- Entropy Divergence ----
def attention_entropy(attn_map):
    p = attn_map.clamp(min=1e-9)
    return -(p * p.log()).sum(dim=-1).mean(dim=-1)  # [batch_size, heads]

def compute_entropy_divergence(attn_maps):
    last_layer = attn_maps[-1]  # Shape: [B, H, T, T]
    if last_layer.dim() != 4:
        raise ValueError(f"Expected attention map with 4 dims [B, H, T, T], got {last_layer.shape}")

    p = last_layer.clamp(min=1e-9)  # Avoid log(0)
    entropy = -(p * p.log()).sum(dim=-1)  # [B, H, T]
    entropy = entropy.mean(dim=-1)       # [B, H] — mean over query positions

    return entropy.mean().item(), entropy.std().item()  # mean and std across batch & heads

# ---- Utility: Decode sample tokens to readable form ----
def decode(indices):
    return " ".join(ID_TO_TOKEN.get(i, "<UNK>") for i in indices)

def decode_triplets(indices):
    triplets = [(indices[i], indices[i+1], indices[i+2]) for i in range(0, len(indices), 3)]
    return [tuple(ID_TO_TOKEN.get(idx, "<UNK>") for idx in triplet) for triplet in triplets]

def decode_target_color(indices):
    return [ID_TO_TOKEN.get(i, "<UNK>") for i in indices]

# ---- Training Loop ----
def train_model(model, optimizer, criterion, dataloader, model_name="baseline", aux_weight=1.0):
    model.train()
    all_loss        = []
    all_entropy     = []
    all_entropy_std = []
    all_ortho_loss  = []
    correct, total  = 0, 0
    for x, y in dataloader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        
        if model_name == "infomax":
            output, attn_maps, layer_outputs = model(x)
            color_logits  = output[:, 0::3, :]     # select every 3rd token in sequence
            loss_main     = criterion(color_logits.reshape(-1, NUM_CLASSES), y.view(-1))
            aux_loss      = model.compute_auxiliary_losses(attn_maps, layer_outputs, entropy_weight=aux_weight, orthogonality_weight=aux_weight)
            loss          = loss_main + aux_loss
            mean_H, std_H = compute_entropy_divergence(attn_maps)
            all_entropy.append(mean_H)
            all_entropy_std.append(std_H)
            ortho_loss    = model.compute_orthogonality_loss(layer_outputs[0])
        else:
            output, attn_maps = model(x)
            color_logits   = output[:, 0::3, :]     # select every 3rd token in sequence
            loss           = criterion(color_logits.reshape(-1, NUM_CLASSES), y.view(-1))
            mean_H, std_H  = compute_entropy_divergence(attn_maps)
            all_entropy.append(mean_H)
            all_entropy_std.append(std_H)

        loss.backward()
        optimizer.step()
        all_loss.append(loss.item())

        if model_name == "infomax":
            all_ortho_loss.append(ortho_loss.item())
        else:
            all_ortho_loss.append(0.0)

        preds    = color_logits.argmax(dim=-1)
        correct += (preds == y).sum().item()
        total   += y.numel()

    return {
        "loss": np.mean(all_loss),
        "acc": correct / total,
        "entropy_mean": np.mean(all_entropy),
        "entropy_std": np.mean(all_entropy_std),
        "ortho": np.mean(all_ortho_loss),
    }


# ---- Validation Loop ----
def eval_model(model, criterion, dataloader, model_name="baseline", aux_weight=1.0):
    model.eval()
    all_loss        = []
    all_entropy     = []
    all_entropy_std = []
    all_ortho_loss  = []
    correct, total  = 0, 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            if model_name == "infomax":
                output, attn_maps, layer_outputs = model(x)
                color_logits = output[:, 0::3, :]     # select every 3rd token in sequence
                loss_main = criterion(color_logits.reshape(-1, NUM_CLASSES), y.view(-1))
                aux_loss = model.compute_auxiliary_losses(attn_maps, layer_outputs, entropy_weight=aux_weight, orthogonality_weight=aux_weight)
                loss = loss_main + aux_loss
                mean_H, std_H = compute_entropy_divergence(attn_maps)
                all_entropy.append(mean_H)
                all_entropy_std.append(std_H)
                ortho_loss    = model.compute_orthogonality_loss(layer_outputs[0])
            else:
                output, attn_maps = model(x)
                color_logits = output[:, 0::3, :]     # select every 3rd token in sequence
                loss = criterion(color_logits.reshape(-1, NUM_CLASSES), y.view(-1))
                mean_H, std_H = compute_entropy_divergence(attn_maps)
                all_entropy.append(mean_H)
                all_entropy_std.append(std_H)

            all_loss.append(loss.item())
            
            if model_name == "infomax":
                all_ortho_loss.append(ortho_loss.item())
            else:
                all_ortho_loss.append(0.0)
            
            preds = color_logits.argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.numel()

    return {
        "loss": np.mean(all_loss),
        "acc": correct / total,
        "entropy_mean": np.mean(all_entropy),
        "entropy_std": np.mean(all_entropy_std),
        "ortho": np.mean(all_ortho_loss),
    }


# ---- Main Training Routine ----
def run_experiment():
    baseline = BaselineTransformer(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, FF_DIM, DROPOUT).to(DEVICE)
    infomax  = InfoMaxTransformer(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, FF_DIM, DROPOUT).to(DEVICE)
    
    opt_base = torch.optim.Adam(baseline.parameters(), lr=1e-4)
    opt_info = torch.optim.Adam(infomax.parameters(), lr=1e-4)
    
    criterion = nn.CrossEntropyLoss()
    
    history_base = defaultdict(list)
    history_info = defaultdict(list)

    print("VOCAB ID RANGES:")
    print("Colors:", [TOKEN_TO_ID[c] for c in VOCAB['colors']])
    print("Shapes:", [TOKEN_TO_ID[s] for s in VOCAB['shapes']])
    print("Positions:", [TOKEN_TO_ID[p] for p in VOCAB['positions']])
    
    for epoch in range(NUM_EPOCHS):
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
        
        tr_base  = train_model(baseline, opt_base, criterion, train_loader, model_name="baseline")
        val_base = eval_model(baseline, criterion, val_loader, model_name="baseline")
        
        for k, v in tr_base.items():
            history_base[f"train_{k}"].append(v)
        for k, v in val_base.items():
            history_base[f"val_{k}"].append(v)
        
        tr_info  = train_model(infomax, opt_info, criterion, train_loader, model_name="infomax", aux_weight=1.0 - epoch/NUM_EPOCHS)
        val_info = eval_model(infomax, criterion, val_loader, model_name="infomax", aux_weight=1.0 - epoch/NUM_EPOCHS)
        
        for k, v in tr_info.items():
            history_info[f"train_{k}"].append(v)
        for k, v in val_info.items():
            history_info[f"val_{k}"].append(v)

        # Print diagnostic metrics
        print("[BASELINE] Loss: {:.4f}  Acc: {:.2%}  Ortho: {:.3f}  Entropy: {:.3f} ± {:.3f}".format(
            val_base['loss'], val_base['acc'], val_base['ortho'], val_base['entropy_mean'], val_base['entropy_std']))
        print("[INFOMAX ] Loss: {:.4f}  Acc: {:.2%}  Ortho: {:.3f}  Entropy: {:.3f} ± {:.3f}".format(
            val_info['loss'], val_info['acc'], val_info['ortho'], val_info['entropy_mean'], val_info['entropy_std']))

        # ---- qualitative sample ----
        sample, target = next(iter(val_loader))
        sample, target = sample.to(DEVICE), target.to(DEVICE)
        
        print("RAW input:", sample[0].tolist())
        print("TRIPLETS:", decode_triplets(sample[0].tolist()))
        
        base_logits = baseline(sample)[0]
        info_logits = infomax(sample)[0]

        base_pred = base_logits[:, 0::3, :].argmax(-1)[0].tolist()
        info_pred = info_logits[:, 0::3, :].argmax(-1)[0].tolist()
        
        print(" input   :", decode_triplets(sample[0].tolist()))
        print(" target  :", decode_target_color(target[0].tolist()))
        print(" baseline:", decode_target_color(base_pred))
        print(" infomax :", decode_target_color(info_pred))
        
        # Visualization
        _, infomax_attn, _ = infomax(sample)
        if epoch < NUM_EPOCHS - 1:
            visualize_attention_heads(infomax_attn[-1], sample_idx=0, layer_idx=NUM_LAYERS - 1, save_path=SAVE_PATH/f"infomax_attn_epoch{epoch+1}.png")
        else:
            visualize_attention_heads(infomax_attn[-1], sample_idx=0, layer_idx=NUM_LAYERS - 1, save_path=SAVE_PATH/f"infomax_attn_epoch{epoch+1}.png")
            
            plot_metrics(history_base, title="Baseline Transformer", save_path=SAVE_PATH/f"baseline_epoch{epoch+1}.png")
            plot_metrics(history_info, title="InfoMax Transformer", save_path=SAVE_PATH/f"infomax_epoch{epoch+1}.png")
    
    return history_base, history_info

# Run training
if __name__ == '__main__':
    run_experiment()
